In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tldextract
from urllib.parse import urlparse
import requests
import time
import pytz
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
timezone = pytz.timezone('Europe/Moscow')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/baitblock-joint-model-dataset/BaitBlockJointDataset.csv
/kaggle/input/baitblock-joint-test-data/BaitBlockJointModelTestDatasetWithURLFeatures.csv


In [4]:
!pip install tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 2.5 MB/s eta 0:00:00a 0:00:01


In [7]:
BAITBLOCK_DS_PATH = "/kaggle/input/baitblock-joint-model-dataset/BaitBlockJointDataset.csv"
WRITE_PATH = "/kaggle/working"

In [6]:
def readCSV(file_path):
    return pd.read_csv(file_path)

def saveCSV(df, file_path):
    df.to_csv(file_path, index=False)
    print(f"DataFrame successfully saved to {file_path}")

In [38]:
bert_df = readCSV(BAITBLOCK_DS_PATH)

In [48]:
bert_df.shape

(4897, 4)

In [46]:
train_val_df, val_df = train_test_split(bert_df, test_size=0.2, random_state=42)
train_df, bert_df = train_test_split(train_val_df, test_size=0.125, random_state=42)
del train_df, train_val_df

In [50]:
saveCSV(bert_df, WRITE_PATH + "/BaitBlockJointModelTestDataset.csv")

DataFrame successfully saved to /kaggle/working/BaitBlockJointModelTestDataset.csv


In [51]:
tld_legitimacy = {
    'com': 1.0,  # Highly legitimate
    'org': 1.0,  # Highly legitimate
    'net': 1.0,  # Generally legitimate
    'edu': 1.0,  # Very high legitimacy
    'gov': 1.0,  # Very high legitimacy
    'mil': 1.0,  # Very high legitimacy
    'info': 0.7, # Moderate legitimacy
    'biz': 0.7,  # Moderate legitimacy
    'name': 0.7, # Moderate legitimacy
    'xyz': 0.4,  # Low legitimacy
    'top': 0.4,  # Low legitimacy
    'club': 0.4, # Low legitimacy
    'online': 0.4, # Low legitimacy
    'work': 0.4, # Low legitimacy
    'tk': 0.1,   # Very low legitimacy
    'cf': 0.1,   # Very low legitimacy
    'gq': 0.1,   # Very low legitimacy
    'ml': 0.1    # Very low legitimacy
}

def get_tld_legitimate_prob(url):
    tld = tldextract.extract(url).suffix
    return tld_legitimacy.get(tld, 0.0) 

In [52]:
def count_redirects(url, timeout=10):
    try:
        response = requests.get(url, allow_redirects=True, timeout=timeout)
        # The number of redirects is the length of the history list
        num_redirects = len(response.history)
        return num_redirects
    except requests.RequestException as e:
        print(f"{url} --> error")
        return 0

In [57]:
def extract_features(url):
    parsed_url = urlparse(url)
    domain = tldextract.extract(url)
    url_length = len(url)
    domain_length = len(domain.domain)
    is_domain_ip = int(parsed_url.netloc.replace('.', '').isdigit())
    tld_legitimate_prob = get_tld_legitimate_prob(url)  
    has_obfuscation = int(any(char in url for char in ['%', '&', '?', '#']))
    is_https = int(parsed_url.scheme == 'https')
    has_external_form_submit = int('submit' in url)
    no_of_url_redirect = count_redirects(url)
    
    result = pd.Series({
        'URLLength': int(url_length),
        'DomainLength': int(domain_length),
        'IsDomainIP': int(is_domain_ip),
        'TLDLegitimateProb': tld_legitimate_prob, 
        'HasObfuscation': int(has_obfuscation),
        'IsHTTPS': int(is_https),
        'HasExternalFormSubmit': int(has_external_form_submit),
        'NoOfURLRedirect': int(no_of_url_redirect)
    })
    
    return result

Initialize progress tracking
start_time = time.time()
total_rows = len(bert_df[bert_df['isURL'] == 1])
processed_rows = 0

def update_progress(url):
    global processed_rows
    global start_time
    features = extract_features(url)
    processed_rows += 1
    progress_percentage = (processed_rows / total_rows) * 100
    elapsed_time = time.time() - start_time
    average_time_per_row = elapsed_time / processed_rows
    remaining_rows = total_rows - processed_rows
    estimated_time_remaining = average_time_per_row * remaining_rows
    estimated_completion_time_utc = datetime.utcnow() + timedelta(seconds=estimated_time_remaining)
    estimated_completion_time_local = estimated_completion_time_utc.replace(tzinfo=pytz.utc).astimezone(timezone)
    print(f"Processed: {processed_rows}/{total_rows} ({progress_percentage:.2f}%)")
    print(f"Estimated completion time (UTC+3): {estimated_completion_time_local.strftime('%Y-%m-%d %H:%M:%S')}")
    return features

features_df = bert_df[bert_df['isURL'] == 1]['url'].apply(update_progress)
features_df = pd.DataFrame(features_df, index=bert_df[bert_df['isURL'] == 1].index)
feature_columns = ['URLLength', 'DomainLength', 'IsDomainIP', 'TLDLegitimateProb', 
                   'HasObfuscation', 'IsHTTPS', 'HasExternalFormSubmit', 'NoOfURLRedirect']
df_features = bert_df.copy()
df_features[feature_columns] = features_df.reindex(bert_df.index)
print(df_features)

       label                                               text  \
20737      1  reProducts that can improve you life, Dear cus...   
13219      0  CBS Fires Don Imus, Larry King Live at 900 pm ...   
13942      0  problem with readline, I have a script that qu...   
35066      1  Re R Query about RODBC to access MySQL from Wi...   
1013       0  Last chance to supercharge your performance, U...   
...      ...                                                ...   
24071      1  Short 30 second form, Thank you for your loan ...   
40272      0  Join VMware for a Full Day of Virtualization w...   
45620      1  More sperm means longer orgasms, Every man wan...   
778        0  R Errors with systemfit package and systemfitC...   
38088      0  Did u find out what time the bus is at coz i n...   

                                                     url  isURL  URLLength  \
20737                                                NaN      0        NaN   
13219                       http://www.

In [31]:
saveCSV(df_features,WRITE_PATH + "/BaitBlockJointModelTestDatasetWithURLFeatures.csv")

DataFrame successfully saved to /kaggle/working/BaitBlockJointModelTestDatasetWithURLFeatures.csv


In [12]:
df_features = readCSV("/kaggle/input/baitblock-joint-test-data/BaitBlockJointModelTestDatasetWithURLFeatures.csv")

In [13]:
from sklearn.preprocessing import StandardScaler

# Assuming these are your numerical features
numerical_columns = ['URLLength', 'DomainLength', 'TLDLegitimateProb']
scaler = StandardScaler()

df_features.loc[:, numerical_columns] = scaler.fit_transform(df_features[numerical_columns])

In [28]:
print(df_features.columns.values)

['label' 'text' 'url' 'isURL' 'URLLength' 'DomainLength' 'IsDomainIP_1'
 'TLDLegitimateProb' 'HasObfuscation_1' 'IsHTTPS_1'
 'HasExternalFormSubmit_1' 'NoOfURLRedirect']


In [27]:
df_features = df_features.rename(columns={'IsDomainIP': 'IsDomainIP_1', 'HasObfuscation': 'HasObfuscation_1', 'IsHTTPS' : 'IsHTTPS_1', 'HasExternalFormSubmit' : 'HasExternalFormSubmit_1'})

In [ ]:
df_features['URL_Domain_Interaction'] = df_features['URLLength'] * df_features['DomainLength']
df_features

In [30]:
df_features = df_features.drop(['URLLength', 'DomainLength'], axis=1)
df_features

,label,text,url,isURL,IsDomainIP_1,TLDLegitimateProb,HasObfuscation_1,IsHTTPS_1,HasExternalFormSubmit_1,NoOfURLRedirect,URL_Domain_Interaction
0,1,"reProducts that can improve you life, Dear cus...",NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,"CBS Fires Don Imus, Larry King Live at 900 pm ...",http://www.cnn.com/larryking,1,0.0,0.737253,0.0,0.0,0.0,2.0,0.714670
2,0,"problem with readline, I have a script that qu...",https://rt.company.com/,1,0.0,0.737253,0.0,1.0,0.0,0.0,0.035455
3,1,Re R Query about RODBC to access MySQL from Wi...,http://www.stats.ox.ac.uk/~ripley/,1,0.0,-1.362906,0.0,0.0,0.0,1.0,0.479383
4,0,"Last chance to supercharge your performance, U...",http://www.promfore.com/?ijoidqngmmtj,1,0.0,0.737253,1.0,0.0,0.0,0.0,-0.045562
...,...,...,...,...,...,...,...,...,...,...,...
4892,1,"Short 30 second form, Thank you for your loan ...",http://krncothesandhairr.com/,1,0.0,0.737253,0.0,0.0,0.0,0.0,-1.566983
4893,0,Join VMware for a Full Day of Virtualization w...,http://info.vmware.com/content/VirtualizationF...,1,0.0,0.737253,1.0,0.0,0.0,1.0,-1.984506
4894,1,"More sperm means longer orgasms, Every man wan...",http://badetall.com,1,0.0,0.737253,0.0,0.0,0.0,0.0,-0.247678
4895,0,R Errors with systemfit package and systemfitC...,https://stat.ethz.ch/mailman/listinfo/r-help,1,0.0,-1.362906,0.0,1.0,0.0,0.0,-0.122225
